# Pretrained baselines: wav2vec 2.0 + CTC, Canary, Parakeet TDT, Qwen3-ASR

COE025 Natural Language Processing, Project 3, Group 8.

Four published checkpoints that the course lists as baselines are run here on the same clips, references, normalisation and scoring as every other system of the project. Nothing is trained and no setting is chosen: each model is run the way its model card and the documentation of `transformers` describe it.

**How to use it.** On Google Colab choose **Runtime > Change runtime type** and a GPU (the committed runs of this project used an NVIDIA L4; Qwen3-ASR runs in bfloat16, which the older T4 does not support well). Run section 0, then **only the section of your own system**. Every section is complete in itself: packages, checkpoint, validation split, test split, result archive. No file has to be uploaded.

| Section | System | Checkpoint | Download |
|---|---|---|---|
| 1 | `wav2vec2-xlsr-53-hungarian` | `jonatasgrosman/wav2vec2-large-xlsr-53-hungarian` | 1.3 GB |
| 2 | `canary-1b-v2` | `nvidia/canary-1b-v2` | 3.9 GB |
| 3 | `parakeet-tdt-0.6b-v3` | `nvidia/parakeet-tdt-0.6b-v3` | 2.5 GB |
| 4 | `qwen3-asr-1.7b` | `Qwen/Qwen3-ASR-1.7B-hf` | 4.1 GB |

**Packages.** All four run through `transformers` 5.23.0, so they need the same pinned packages (`requirements-pretrained.txt`). Nothing conflicts, the sections can be run in any order in one runtime, and no section needs a fresh runtime. The packages `nemo_toolkit` and `qwen-asr`, which two model cards name first, pin other versions and are not installed.

**This notebook runs the test split, and it is the only notebook of the project that does.** Each system is run on it once. The reason: these are published checkpoints with nothing chosen by us on any data, so there is nothing that a look at the test split could tune, and the test evaluation of the project's other systems has already taken place (22 November 2026). The runner still demands the flag of the final evaluation and refuses to replace an existing test run. If a test cell stops because a run already exists, that system has been evaluated: do not run it again.

Every setting is fixed in `src/huasr/pretrained.py`; the cells only call that code. The times given below are estimates, not measurements.

## 0. Setup (run this first, whatever your system is)

In [ ]:
# Clone the public repository: code and audio (about 1.3 GB). Nothing is uploaded by hand.
import os

REPOSITORY = "https://github.com/kaankalkan03/NLP-project.git"
BRANCH = "main"
FOLDER = "/content/NLP-project"

if not os.path.isdir(FOLDER):  # running this cell again keeps the existing clone
    !git clone --depth 1 --branch {BRANCH} {REPOSITORY} {FOLDER}
%cd {FOLDER}
!git log -1 --format="commit %H  %cd"

In [ ]:
# A failed command must not be followed by more work on top of it: this stops "Run all".
def stop_unless_done(step):
    assert globals().get("_exit_code", 0) == 0, f"{step} failed: read the message above before going on"

`pip` may report that other Colab packages want different versions of numpy, pyarrow or requests; those packages are not used here. If Colab offers to restart the session, that is not needed: every step below runs in a new Python process. (After a restart, run the first two cells again.)

In [ ]:
# The pinned packages of the repository, then the package itself.
!pip install -q -r requirements.txt
!pip install -q --no-deps -e .

In [ ]:
# The GPU, and the four systems with checkpoint, revision and settings; every run writes them into its record.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -m huasr.pretrained list

In [ ]:
# Audio: every clip is restored from the tar shards and checked against the SHA-256 in the manifest.
# The runner reads the shards itself; this cell shows that the clone holds all of the audio.
!python -m huasr.subset unpack --raw data/raw/voxpopuli
stop_unless_done("unpack")

In [ ]:
# Unit tests of the four systems, with stand-ins for the models: nothing is downloaded or loaded.
!python -m unittest tests.test_pretrained 2>&1 | tail -n 4

## 1. wav2vec 2.0 + CTC: `wav2vec2-xlsr-53-hungarian`

**Checkpoint:** `jonatasgrosman/wav2vec2-large-xlsr-53-hungarian` at revision `2bd0786e71cad097c4e6ce9b3e45b40c74ad8f37`, licence Apache-2.0. Its model card: XLSR-53 fine-tuned for Hungarian on the train and validation splits of Common Voice 6.1 and on CSS10; input at 16 kHz.

**Language:** the checkpoint is Hungarian only; there is nothing to set. **Decoding:** as in the script of the model card, the best token of every frame and no language model. The model writes upper-case letters without punctuation; case is left to the scoring policy. **Long clips:** the model has no window; every clip is given to it whole.

Expected time: a few minutes per split.

In [ ]:
# Packages of this system (the same pinned set for all four) and its checkpoint (1.3 GB, checked against a pinned SHA-256).
!pip install -q -r requirements-pretrained.txt
!python -m huasr.pretrained fetch wav2vec2-xlsr-53-hungarian
stop_unless_done("the download of the checkpoint")

In [ ]:
# Validation split: run every clip, then score.
!python -m huasr.evaluate --system wav2vec2-xlsr-53-hungarian --force
stop_unless_done("the validation run")
!python -m huasr.score results/baselines/wav2vec2-xlsr-53-hungarian/validation
stop_unless_done("scoring")

In [ ]:
# Test split, once: the flag marks the final evaluation, and an existing test run is not replaced.
!python -m huasr.evaluate --system wav2vec2-xlsr-53-hungarian --split test --final-test
stop_unless_done("the test run")
!python -m huasr.score results/baselines/wav2vec2-xlsr-53-hungarian/test
stop_unless_done("scoring")

In [ ]:
# Pack the result folders of this system (run records, predictions, scores) and download the archive.
ARCHIVE = "/content/wav2vec2-xlsr-53-hungarian_results.zip"
!rm -f {ARCHIVE}
!zip -q -r {ARCHIVE} results/baselines/wav2vec2-xlsr-53-hungarian
!unzip -l {ARCHIVE}

from google.colab import files
files.download(ARCHIVE)

## 2. Canary (attention encoder-decoder): `canary-1b-v2`

**Checkpoint:** `nvidia/canary-1b-v2` at revision `d455706339a6b32e1aa40f82c713a482a0c938e2`, licence CC BY 4.0. Its model card lists Hungarian among 25 languages.

**Language:** source language `hu` and target language `hu`. Equal languages are how Canary is told to transcribe and not to translate; the code checks the decoder prompt of every clip for the Hungarian token twice. **Decoding:** greedy, as the checkpoint sets it. **Long clips:** Canary is used on at most 40 seconds (beyond that its own toolkit cuts the audio too); a longer clip is cut into the fewest equal parts and the texts are joined. Two validation clips and one test clip are longer.

Its model card lists the Granary corpus among its training data; whether that corpus includes the recordings of our test split has not been checked. Expected time: 15 to 30 minutes per split.

In [ ]:
# Packages of this system (the same pinned set for all four) and its checkpoint (3.9 GB, checked against a pinned SHA-256).
!pip install -q -r requirements-pretrained.txt
!python -m huasr.pretrained fetch canary-1b-v2
stop_unless_done("the download of the checkpoint")

In [ ]:
# Validation split: run every clip, then score.
!python -m huasr.evaluate --system canary-1b-v2 --force
stop_unless_done("the validation run")
!python -m huasr.score results/baselines/canary-1b-v2/validation
stop_unless_done("scoring")

In [ ]:
# Test split, once: the flag marks the final evaluation, and an existing test run is not replaced.
!python -m huasr.evaluate --system canary-1b-v2 --split test --final-test
stop_unless_done("the test run")
!python -m huasr.score results/baselines/canary-1b-v2/test
stop_unless_done("scoring")

In [ ]:
# Pack the result folders of this system (run records, predictions, scores) and download the archive.
ARCHIVE = "/content/canary-1b-v2_results.zip"
!rm -f {ARCHIVE}
!zip -q -r {ARCHIVE} results/baselines/canary-1b-v2
!unzip -l {ARCHIVE}

from google.colab import files
files.download(ARCHIVE)

## 3. Parakeet TDT (transducer): `parakeet-tdt-0.6b-v3`

**Checkpoint:** `nvidia/parakeet-tdt-0.6b-v3` at revision `541d1f99c6b0c3cd0b11a95167540bb8edefd82b`, licence CC BY 4.0. Its model card lists Hungarian among 25 languages.

**Language:** the model detects the language itself and takes no language setting (model card), so none can be wrong or missing; the run record says so. **Decoding:** greedy TDT, as the checkpoint sets it. **Long clips:** the model reads up to 24 minutes with full attention (model card); every clip is given to it whole.

Its model card lists the Granary corpus among its training data, as for Canary; whether that corpus includes the recordings of our test split has not been checked. Expected time: 15 to 30 minutes per split.

In [ ]:
# Packages of this system (the same pinned set for all four) and its checkpoint (2.5 GB, checked against a pinned SHA-256).
!pip install -q -r requirements-pretrained.txt
!python -m huasr.pretrained fetch parakeet-tdt-0.6b-v3
stop_unless_done("the download of the checkpoint")

In [ ]:
# Validation split: run every clip, then score.
!python -m huasr.evaluate --system parakeet-tdt-0.6b-v3 --force
stop_unless_done("the validation run")
!python -m huasr.score results/baselines/parakeet-tdt-0.6b-v3/validation
stop_unless_done("scoring")

In [ ]:
# Test split, once: the flag marks the final evaluation, and an existing test run is not replaced.
!python -m huasr.evaluate --system parakeet-tdt-0.6b-v3 --split test --final-test
stop_unless_done("the test run")
!python -m huasr.score results/baselines/parakeet-tdt-0.6b-v3/test
stop_unless_done("scoring")

In [ ]:
# Pack the result folders of this system (run records, predictions, scores) and download the archive.
ARCHIVE = "/content/parakeet-tdt-0.6b-v3_results.zip"
!rm -f {ARCHIVE}
!zip -q -r {ARCHIVE} results/baselines/parakeet-tdt-0.6b-v3
!unzip -l {ARCHIVE}

from google.colab import files
files.download(ARCHIVE)

## 4. Qwen3-ASR: `qwen3-asr-1.7b`

**Checkpoint:** `Qwen/Qwen3-ASR-1.7B-hf` at revision `bcd2b5b7f32b480ab5790554cfa8347f246a14f3`, licence Apache-2.0. This is the copy of `Qwen/Qwen3-ASR-1.7B` that Qwen publishes in the Transformers format; the original repository can only be loaded with the package `qwen-asr`, which pins other versions than this project runs on. The 1.7B model takes about 4 GB of GPU memory in bfloat16, so the smaller 0.6B model is not needed. Its model card lists Hungarian among 30 languages.

**Language:** Hungarian is forced with the prefix the model was trained with, `language Hungarian<asr_text>`; the code checks the prompt of every clip for it. **Decoding:** greedy, at most 1,024 new tokens. The repetition clean-up that the processor offers is not applied: the hypothesis is the decoded tokens, as for every other system. **Long clips:** the model handles long audio (model card); every clip is given to it whole.

The model card does not list the training data. Expected time: 30 to 60 minutes per split.

In [ ]:
# Packages of this system (the same pinned set for all four) and its checkpoint (4.1 GB, checked against a pinned SHA-256).
!pip install -q -r requirements-pretrained.txt
!python -m huasr.pretrained fetch qwen3-asr-1.7b
stop_unless_done("the download of the checkpoint")

In [ ]:
# Validation split: run every clip, then score.
!python -m huasr.evaluate --system qwen3-asr-1.7b --force
stop_unless_done("the validation run")
!python -m huasr.score results/baselines/qwen3-asr-1.7b/validation
stop_unless_done("scoring")

In [ ]:
# Test split, once: the flag marks the final evaluation, and an existing test run is not replaced.
!python -m huasr.evaluate --system qwen3-asr-1.7b --split test --final-test
stop_unless_done("the test run")
!python -m huasr.score results/baselines/qwen3-asr-1.7b/test
stop_unless_done("scoring")

In [ ]:
# Pack the result folders of this system (run records, predictions, scores) and download the archive.
ARCHIVE = "/content/qwen3-asr-1.7b_results.zip"
!rm -f {ARCHIVE}
!zip -q -r {ARCHIVE} results/baselines/qwen3-asr-1.7b
!unzip -l {ARCHIVE}

from google.colab import files
files.download(ARCHIVE)

## After your section: commit the results

Unpack the archive in the root of your own clone of the repository. It holds `results/baselines/<system>/validation/` and `results/baselines/<system>/test/`, each with `run.json`, `predictions.csv` and `scores.json`. Then:

```bash
python -m huasr.report      # adds the rows of your system to the tables in results/README.md
python -m unittest          # includes the check that the READMEs are up to date
git add results/baselines/<system> results/README.md
git commit -m "Add <system> on the validation and the test split"
```

Commit from your own GitHub account. No model weights are in the archive, and none may be added to the repository.